# **Build a Chatbot using Hugging Face Transformers.**

In [ ]:
!pip install transformers torch --quiet

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

#load tokenizer and model used for the chatbot trasnformers.
model_name="microsoft/DialoGPT-medium"

tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token=tokenizer.eos_token

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Write the response function by chatbot.
def generate_response(user_input, chat_history_ids=None):

    # user input and add end sequence token.
    new_user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # append new user input to chat
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
    else:
        bot_input_ids = new_user_input_ids

    # added attention mask.to get reliable outputs..
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # generate response.
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.90,
        temperature=0.70,          # good balance
        no_repeat_ngram_size=3,
        min_length=5,
        repetition_penalty=1.2     # helps reduce repetition
    )

    # Decode only the newly generated tokens at time.
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [ ]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

chat_history_ids = None

while True:
    raw_input = input("User: ").strip()

    # Improved stop condition - checks multiple variations
    if raw_input.lower() in ["exit", "quit", "bye", "goodbye", "stop", "end"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # Call the model with clean user input
    response, chat_history_ids = generate_response(raw_input, chat_history_ids)

    # Strong post-processing to clean bad outputs
    response = response.strip()

    # Remove any unwanted parts that the model sometimes adds
    if "User:" in response:
        response = response.split("User:")[0].strip()
    if "Bot:" in response or "chatbot:" in response.lower():
        response = response.split("Bot:")[-1].split("chatbot:")[-1].strip()

    # Fallback if response is empty, too short, or garbage
    if not response or len(response) < 4 or response.lower() in ["", " ", ".", "..", "hmm"]:
        response = "Sorry, I didn't catch that. Could you please rephrase your question?"

    print(f"Chatbot: {response}")

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: what is artificial intelligence
Chatbot: I was thinking the same thing. What's the deal? I've seen it before, but never heard of anyone doing this.
User: Thank You.
Chatbot: No problem, glad to help!
User: exit
Chatbot: Goodbye! Have a great day.


**User:** What is Artificial Intelligence?  
**Chatbot:** [Expected good answer]

**User:** Who created Python?  
**Chatbot:** [Expected good answer]

**User:** Thank you  
**Chatbot:** You're welcome!

**User:** exit  
**Chatbot:** Goodbye! Have a great day.